In [90]:
# Install Required Libraries
!pip install -q langchain-groq langchain-core python-dotenv

In [91]:
#Setup API Keys
import os
from google.colab import userdata


os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

print("Keys loaded")

Keys loaded


In [92]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Initialize the Groq LLM with a CURRENT model
# llama-3.3-70b-versatile is currently one of their best and most stable
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# 2. Define the Extraction Instruction
extract_prompt = ChatPromptTemplate.from_template("""
Extract professional skills, tools, and years of experience from this resume.
Resume: {resume_text}
Constraint: Do NOT assume skills not explicitly mentioned.
""")

# 3. Define the Scoring Instruction
score_prompt = ChatPromptTemplate.from_template("""
Compare the Extracted Details with the Job Description.
Job Description: {job_description}
Extracted Details: {extracted_details}

Output a JSON object with these exact keys:
"fit_score": (integer 0-100),
"explanation": (reasoning),
"missing_skills": (list)
""")

# 4. Build the Chain (The Pipeline)
pipeline = (
    {
        "extracted_details": extract_prompt | llm,
        "job_description": lambda x: x["job_description"]
    }
    | score_prompt
    | llm
    | JsonOutputParser()
)

print("Pipeline logic updated with the new model!")

Pipeline logic updated with the new model!


In [93]:
# Define your Job Description
jd = "Python Developer: Requires Python, SQL, and 3+ years experience in Backend development."

# Define your 3 test cases
candidates = [
    {"name": "Strong", "text": "Senior Backend Dev with 5 years experience. Expert in Python and SQL. Built scalable APIs."},
    {"name": "Average", "text": "Junior Dev with 1 year experience. Knows Python but very little SQL. Fast learner."},
    {"name": "Weak", "text": "Professional Yoga Instructor with 8 years experience in wellness and mindfulness."}
]

for person in candidates:
    print(f"--- Processing {person['name']} Candidate ---")
    try:
        response = pipeline.invoke({
            "resume_text": person['text'],
            "job_description": jd
        })
        print(f"Score: {response['fit_score']}")
        print(f"Reason: {response['explanation']}\n")
    except Exception as e:
        print(f"An error occurred: {e}")

--- Processing Strong Candidate ---
Score: 100
Reason: The extracted details match all the requirements mentioned in the job description. The candidate has 5 years of experience, which exceeds the required 3+ years of experience in Backend development. The candidate also possesses the required skills in Python and SQL.

--- Processing Average Candidate ---
Score: 40
Reason: The candidate has some relevant skills, including Python programming and limited knowledge of SQL. However, they fall short of the required 3+ years of experience in backend development, having only 1 year of experience. This partial match is the reason for the moderate fit score.

--- Processing Weak Candidate ---
Score: 0
Reason: The extracted details do not match the job description. The job requires Python, SQL, and 3+ years of experience in Backend development, but the extracted details mention yoga instruction, wellness, and mindfulness as professional skills, with no mention of Python, SQL, or backend develop